In [ ]:
# Traditional Chinese Music Dataset Exploration
# Interactive analysis of the ccmusic-database-demo for music therapy research

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import librosa
import soundfile as sf
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🎵 Traditional Chinese Music Dataset Exploration")
print("=" * 50)

# ## 1. Dataset Loading and Initial Exploration

# Load the recommendation system
import sys
sys.path.append('../src')
from src.tcm_recommender import CCMusicDatasetLoader, TCMRecommendationEngine

# Initialize dataset loader
dataset_path = "../data/ccmusic-database-demo"
print(f"Loading dataset from: {dataset_path}")

try:
    loader = CCMusicDatasetLoader(dataset_path)
    df = loader.load_dataset()
    print(f"✅ Successfully loaded {len(df)} tracks")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Please ensure the ccmusic dataset is downloaded and extracted to the correct path")
    # Create sample data for demonstration
    df = pd.DataFrame({
        'track_id': ['ctis_001', 'ctis_002', 'folk_001'],
        'title': ['Spring River', 'Autumn Moon', 'Folk Song'],
        'era': ['1950s', 'traditional', '1940s'],
        'instruments': [['erhu'], ['guqin'], ['dizi']],
        'mood': ['nostalgic', 'peaceful', 'joyful'],
        'tempo': ['slow', 'slow', 'moderate'],
        'cultural_authenticity_score': [0.9, 0.95, 0.8]
    })
    print("📝 Using sample data for demonstration")

# ## 2. Basic Dataset Statistics

print("\n🔍 Dataset Overview:")
print("-" * 30)
print(f"Total tracks: {len(df)}")
print(f"Unique track IDs: {df['track_id'].nunique()}")
print(f"Missing values per column:")
for col in df.columns:
    missing = df[col].isnull().sum()
    if missing > 0:
        print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")

# Display first few tracks
print(f"\n📋 Sample Tracks:")
display_cols = ['track_id', 'title', 'era', 'mood', 'instruments']
sample_df = df[display_cols].head()
print(sample_df.to_string(index=False))

# ## 3. Era Distribution Analysis

print(f"\n🕐 Historical Era Distribution:")
era_counts = df['era'].value_counts()
print(era_counts)

# Create era distribution visualization
fig_era = px.pie(
    values=era_counts.values,
    names=era_counts.index,
    title="Distribution of Musical Eras in Dataset",
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig_era.show()

# Era timeline visualization
era_order = ['traditional', '1930s', '1940s', '1950s', '1960s']
era_data = []
for era in era_order:
    count = era_counts.get(era, 0)
    era_data.append({'Era': era, 'Count': count, 'Percentage': count/len(df)*100})

era_timeline_df = pd.DataFrame(era_data)
print(f"\n📊 Era Timeline:")
print(era_timeline_df.to_string(index=False))

fig_timeline = px.bar(
    era_timeline_df,
    x='Era',
    y='Count',
    title="Traditional Chinese Music by Historical Era",
    color='Count',
    color_continuous_scale='Blues'
)
fig_timeline.update_layout(xaxis={'categoryorder':'array', 'categoryarray':era_order})
fig_timeline.show()

# ## 4. Musical Characteristics Analysis

# Mood distribution
print(f"\n😊 Mood Distribution:")
mood_counts = df['mood'].value_counts()
print(mood_counts)

# Tempo distribution
print(f"\n🎵 Tempo Distribution:")
tempo_counts = df['tempo'].value_counts()
print(tempo_counts)

# Combined mood and tempo analysis
fig_mood_tempo = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Mood Distribution', 'Tempo Distribution'),
    specs=[[{"type": "domain"}, {"type": "domain"}]]
)

fig_mood_tempo.add_trace(go.Pie(
    labels=mood_counts.index,
    values=mood_counts.values,
    name="Mood"
), row=1, col=1)

fig_mood_tempo.add_trace(go.Pie(
    labels=tempo_counts.index,
    values=tempo_counts.values,
    name="Tempo"
), row=1, col=2)

fig_mood_tempo.update_traces(hole=.4, hoverinfo="label+percent+name")
fig_mood_tempo.update_layout(title_text="Musical Characteristics Distribution")
fig_mood_tempo.show()

# ## 5. Traditional Chinese Instruments Analysis

# Extract and analyze instruments
all_instruments = []
for instruments in df['instruments']:
    if isinstance(instruments, list):
        all_instruments.extend(instruments)
    elif isinstance(instruments, str):
        # Handle comma-separated strings
        all_instruments.extend([inst.strip() for inst in instruments.split(',')])

instrument_counts = pd.Series(all_instruments).value_counts()
print(f"\n🎻 Traditional Chinese Instruments:")
print(instrument_counts.head(10))

# Instrument popularity chart
fig_instruments = px.bar(
    x=instrument_counts.head(10).index,
    y=instrument_counts.head(10).values,
    title="Most Common Traditional Chinese Instruments",
    labels={'x': 'Instrument', 'y': 'Frequency'},
    color=instrument_counts.head(10).values,
    color_continuous_scale='Viridis'
)
fig_instruments.update_layout(xaxis_tickangle=-45)
fig_instruments.show()

# Instrument combinations analysis
if 'instruments' in df.columns:
    combo_analysis = []
    for _, row in df.iterrows():
        instruments = row['instruments']
        if isinstance(instruments, list) and len(instruments) > 1:
            combo = ' + '.join(sorted(instruments))
            combo_analysis.append({
                'track_id': row['track_id'],
                'combination': combo,
                'num_instruments': len(instruments),
                'era': row['era'],
                'mood': row['mood']
            })

    if combo_analysis:
        combo_df = pd.DataFrame(combo_analysis)
        popular_combos = combo_df['combination'].value_counts().head(5)
        print(f"\n🎼 Popular Instrument Combinations:")
        print(popular_combos)

# ## 6. Cultural Authenticity Analysis

if 'cultural_authenticity_score' in df.columns:
    auth_scores = df['cultural_authenticity_score'].dropna()

    print(f"\n🏮 Cultural Authenticity Analysis:")
    print(f"Mean authenticity score: {auth_scores.mean():.3f}")
    print(f"Median authenticity score: {auth_scores.median():.3f}")
    print(f"Standard deviation: {auth_scores.std():.3f}")

    # Authenticity distribution
    fig_auth = px.histogram(
        auth_scores,
        title="Distribution of Cultural Authenticity Scores",
        labels={'value': 'Cultural Authenticity Score', 'count': 'Number of Tracks'},
        nbins=20,
        color_discrete_sequence=['skyblue']
    )
    fig_auth.add_vline(x=auth_scores.mean(), line_dash="dash", line_color="red",
                       annotation_text=f"Mean: {auth_scores.mean():.3f}")
    fig_auth.show()

    # Authenticity by era
    auth_by_era = df.groupby('era')['cultural_authenticity_score'].agg(['mean', 'std', 'count']).round(3)
    print(f"\n📊 Authenticity by Era:")
    print(auth_by_era)

    fig_auth_era = px.box(
        df,
        x='era',
        y='cultural_authenticity_score',
        title="Cultural Authenticity by Historical Era",
        color='era'
    )
    fig_auth_era.update_layout(xaxis={'categoryorder':'array', 'categoryarray':era_order})
    fig_auth_era.show()

# ## 7. Era-Specific Musical Characteristics

# Cross-tabulation analysis
print(f"\n🔄 Era vs Mood Cross-Analysis:")
era_mood_crosstab = pd.crosstab(df['era'], df['mood'], normalize='index') * 100
print(era_mood_crosstab.round(1))

# Heatmap visualization
fig_heatmap = px.imshow(
    era_mood_crosstab.values,
    labels=dict(x="Mood", y="Era", color="Percentage"),
    x=era_mood_crosstab.columns,
    y=era_mood_crosstab.index,
    title="Era vs Mood Distribution (%)",
    color_continuous_scale='Blues'
)
fig_heatmap.show()

# Era vs Tempo analysis
print(f"\n🎵 Era vs Tempo Cross-Analysis:")
era_tempo_crosstab = pd.crosstab(df['era'], df['tempo'], normalize='index') * 100
print(era_tempo_crosstab.round(1))

# ## 8. Track Duration Analysis (if available)

if 'duration' in df.columns:
    durations = df['duration'].dropna()

    print(f"\n⏱️ Track Duration Analysis:")
    print(f"Mean duration: {durations.mean():.1f} seconds ({durations.mean()/60:.1f} minutes)")
    print(f"Median duration: {durations.median():.1f} seconds")
    print(f"Range: {durations.min():.1f} - {durations.max():.1f} seconds")

    # Duration distribution
    fig_duration = px.histogram(
        durations,
        title="Distribution of Track Durations",
        labels={'value': 'Duration (seconds)', 'count': 'Number of Tracks'},
        nbins=20
    )
    fig_duration.add_vline(x=durations.mean(), line_dash="dash", line_color="red",
                          annotation_text=f"Mean: {durations.mean():.0f}s")
    fig_duration.show()

    # Duration by era
    duration_by_era = df.groupby('era')['duration'].agg(['mean', 'median', 'std']).round(1)
    print(f"\n📊 Duration by Era:")
    print(duration_by_era)

# ## 9. Recommendation System Testing

print(f"\n🎯 Testing Recommendation System:")
print("-" * 40)

try:
    # Create recommendation engine
    recommender = TCMRecommendationEngine(df)
    print("✅ Recommendation engine initialized")

    # Test different preference profiles
    test_preferences = [
        {
            'name': 'Elderly 1950s Listener',
            'preferences': {
                'era': '1950s',
                'instruments': ['erhu', 'guqin'],
                'mood': 'nostalgic',
                'tempo': 'slow'
            }
        },
        {
            'name': 'Traditional Music Lover',
            'preferences': {
                'era': 'traditional',
                'instruments': ['guqin'],
                'mood': 'peaceful',
                'tempo': 'slow'
            }
        },
        {
            'name': 'Cultural Bridge (1940s)',
            'preferences': {
                'era': '1940s',
                'instruments': ['erhu', 'pipa'],
                'mood': 'melancholic',
                'tempo': 'moderate'
            }
        }
    ]

    for test_case in test_preferences:
        print(f"\n🔍 {test_case['name']}:")
        preferences = test_case['preferences']

        recommendations = recommender.recommend_tracks(
            preferences,
            num_recommendations=3
        )

        if recommendations:
            for i, track in enumerate(recommendations, 1):
                print(f"  {i}. {track['title']} ({track['era']}) - Score: {track['final_score']:.3f}")
                print(f"     Instruments: {', '.join(track['instruments'])}")
                print(f"     Mood: {track['mood']}, Tempo: {track['tempo']}")
        else:
            print("  No recommendations found")

except Exception as e:
    print(f"❌ Error testing recommendation system: {e}")

# ## 10. Data Quality Assessment

print(f"\n🔍 Data Quality Assessment:")
print("-" * 35)

# Check for duplicates
duplicates = df.duplicated(subset=['title', 'era']).sum()
print(f"Potential duplicate tracks: {duplicates}")

# Check data completeness
completeness_scores = {}
for col in df.columns:
    completeness = (1 - df[col].isnull().sum() / len(df)) * 100
    completeness_scores[col] = completeness

print(f"\n📊 Data Completeness by Column:")
for col, score in sorted(completeness_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {col}: {score:.1f}%")

# Identify tracks suitable for therapy (high cultural authenticity)
if 'cultural_authenticity_score' in df.columns:
    high_auth_tracks = df[df['cultural_authenticity_score'] >= 0.8]
    print(f"\n🏮 High Cultural Authenticity Tracks (≥0.8): {len(high_auth_tracks)}")

    if len(high_auth_tracks) > 0:
        print("Era distribution in high-authenticity tracks:")
        high_auth_eras = high_auth_tracks['era'].value_counts()
        print(high_auth_eras)

# ## 11. Therapeutic Suitability Analysis

print(f"\n💊 Therapeutic Suitability Analysis:")
print("-" * 40)

# Define therapeutic criteria
therapeutic_criteria = {
    'cultural_authenticity': 0.7,  # Minimum authenticity score
    'preferred_eras': ['1930s', '1940s', '1950s', '1960s'],  # Relevant for elderly patients
    'calming_moods': ['nostalgic', 'peaceful'],  # Therapeutic moods
    'suitable_tempos': ['slow', 'moderate']  # Not overstimulating
}

# Filter tracks meeting therapeutic criteria
therapeutic_tracks = df.copy()

if 'cultural_authenticity_score' in df.columns:
    therapeutic_tracks = therapeutic_tracks[
        therapeutic_tracks['cultural_authenticity_score'] >= therapeutic_criteria['cultural_authenticity']
    ]

therapeutic_tracks = therapeutic_tracks[
    therapeutic_tracks['era'].isin(therapeutic_criteria['preferred_eras'])
]

therapeutic_tracks = therapeutic_tracks[
    therapeutic_tracks['mood'].isin(therapeutic_criteria['calming_moods'])
]

therapeutic_tracks = therapeutic_tracks[
    therapeutic_tracks['tempo'].isin(therapeutic_criteria['suitable_tempos'])
]

print(f"Tracks meeting therapeutic criteria: {len(therapeutic_tracks)}/{len(df)} ({len(therapeutic_tracks)/len(df)*100:.1f}%)")

if len(therapeutic_tracks) > 0:
    print(f"\nTherapeutic track characteristics:")
    print(f"Era distribution: {therapeutic_tracks['era'].value_counts().to_dict()}")
    print(f"Mood distribution: {therapeutic_tracks['mood'].value_counts().to_dict()}")
    print(f"Tempo distribution: {therapeutic_tracks['tempo'].value_counts().to_dict()}")

# ## 12. Export Findings for Further Analysis

# Create summary statistics
dataset_summary = {
    'total_tracks': len(df),
    'unique_tracks': df['track_id'].nunique(),
    'era_distribution': df['era'].value_counts().to_dict(),
    'mood_distribution': df['mood'].value_counts().to_dict(),
    'tempo_distribution': df['tempo'].value_counts().to_dict(),
    'instrument_frequency': instrument_counts.head(10).to_dict(),
    'therapeutic_suitable_tracks': len(therapeutic_tracks),
    'data_completeness': completeness_scores
}

if 'cultural_authenticity_score' in df.columns:
    dataset_summary['authenticity_stats'] = {
        'mean': float(df['cultural_authenticity_score'].mean()),
        'median': float(df['cultural_authenticity_score'].median()),
        'std': float(df['cultural_authenticity_score'].std())
    }

# Save summary
output_path = "../data/dataset_exploration_summary.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_summary, f, indent=2, ensure_ascii=False)

print(f"\n💾 Dataset exploration summary saved to: {output_path}")

# Create filtered dataset for therapy research
therapy_dataset_path = "../data/therapeutic_tracks.csv"
if len(therapeutic_tracks) > 0:
    therapeutic_tracks.to_csv(therapy_dataset_path, index=False)
    print(f"💊 Therapeutic tracks dataset saved to: {therapy_dataset_path}")

print(f"\n✅ Dataset exploration complete!")
print(f"📊 Key findings:")
print(f"   - Total tracks analyzed: {len(df)}")
print(f"   - Historical eras covered: {df['era'].nunique()}")
print(f"   - Traditional instruments found: {len(instrument_counts)}")
print(f"   - Therapeutically suitable tracks: {len(therapeutic_tracks)}")
print(f"   - Data quality: {np.mean(list(completeness_scores.values())):.1f}% complete")

# ## Next Steps
print(f"\n🚀 Recommended Next Steps:")
print(f"   1. Review therapeutic tracks in: {therapy_dataset_path}")
print(f"   2. Test AI music generation with high-authenticity tracks")
print(f"   3. Create session playlists balanced by era and mood")
print(f"   4. Validate cultural authenticity with domain experts")
print(f"   5. Begin pilot testing with target participants")